# Cell 1 — Install and imports:



In [1]:
!pip install -q transformers accelerate

import json
import numpy as np
from transformers import pipeline

# Cell 2 — Knowledge base:



In [2]:
DOCUMENTS = [                            # start of the knowledge base list
    "France Q3 sales: 1.2M EUR. Top products: WidgetA, WidgetB.",     # France sales doc
    "Germany Q3 sales: 1.8M EUR. Top products: WidgetB, WidgetC.",    # Germany sales doc
    "Recall notice: WidgetB recalled in EU as of Oct 2025. WidgetC under review.",  # recall doc
    "EU refund policy: 30 days return plus 14-day cooling-off period.",  # EU policy doc
    "US refund policy: 14 days return, no cooling-off period.",          # US policy doc
]                                        # end of knowledge base

# Cell 3 — Embedding and similarity:




In [3]:
def embed(text):                         # define function to turn text into a vector
    vocab = set(" ".join(DOCUMENTS).lower().split())   # build vocabulary from all docs
    vec = np.zeros(len(vocab))           # create a zero vector the size of the vocab
    words = text.lower().split()         # lowercase and split the input text into words
    for i, w in enumerate(sorted(vocab)):  # loop over vocabulary in sorted order
        vec[i] = words.count(w)          # count how many times each word appears
    return vec                           # return the bag-of-words vector

def cosine(a, b):                        # define cosine similarity function
    denom = (np.linalg.norm(a) * np.linalg.norm(b))   # compute product of vector magnitudes
    return float(np.dot(a, b) / denom) if denom else 0.0   # return similarity, or 0 if zero vector

# Cell 4 — Load the LLM:




In [ ]:
llm = pipeline(                          # load the language model
    "text-generation",                   # task type: causal text generation
    model="mistralai/Mistral-7B-Instruct-v0.2",   # model name on Hugging Face
    device_map="auto",                   # automatically place model on GPU or CPU
    dtype="auto",                        # automatically choose numeric precision
)                                        # end of pipeline setup

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

# Cell 5 — Tools the agent can call:




In [ ]:
def search_docs(query):                  # define general search tool
    q_vec = embed(query)                 # embed the incoming query
    scored = [(cosine(q_vec, embed(doc)), doc) for doc in DOCUMENTS]  # score each doc against query
    scored.sort(reverse=True)            # sort docs by similarity, highest first
    return [doc for _, doc in scored[:2]]  # return the top 2 documents

def search_sales(_query):                # define sales-specific search tool
    return search_docs("France Germany Q3 sales")   # search with a fixed sales query

def search_recalls(_query):              # define recall-specific search tool
    return search_docs("recall WidgetB WidgetC EU")  # search with a fixed recall query

TOOLS = {                                # build a registry of tools the agent can call
    "search_docs": search_docs,          # map name to function
    "search_sales": search_sales,        # map name to function
    "search_recalls": search_recalls,    # map name to function
}                                        # end of tool registry

# Cell 6 — Agent planner:




In [ ]:
def plan(query, history):                # define the agent planner
    used = [h["tool"] for h in history]  # list tools already used
    q = query.lower()                    # lowercase the query for matching

    if "sales" in q and "search_sales" not in used:   # if sales mentioned and not yet searched
        return "search_sales"            # choose sales tool
    if "recall" in q and "search_recalls" not in used:  # if recall mentioned and not yet searched
        return "search_recalls"          # choose recall tool
    if "search_docs" not in used:        # if general search not yet done
        return "search_docs"             # choose general search tool
    return None                          # otherwise, signal that the agent is done

# Cell 7 — Agent loop:





In [ ]:
def agentic_rag(query, max_steps=5):     # define the main agent loop
    history = []                         # store each tool result
    trace = []                           # store a readable log of steps

    for step in range(max_steps):        # loop up to max_steps times
        tool_name = plan(query, history)  # ask the planner what to do next

        if tool_name is None:            # if planner says done
            trace.append(f"Step {step+1}: no more tools, synthesizing")  # log it
            break                        # exit the loop

        result = TOOLS[tool_name](query)  # call the chosen tool
        history.append({"tool": tool_name, "result": result})  # store the result
        trace.append(f"Step {step+1}: called {tool_name} -> {result}")  # log the step

    context = "\n".join(f"- {h['result']}" for h in history)  # combine all results into one context
    prompt = (                           # build the final prompt
        "Answer the question using only the context below.\n\n"   # instruction to stay grounded
        f"Context:\n{context}\n\n"       # insert gathered context
        f"Question: {query}\n"           # insert user question
        "Answer:"                        # prompt the model to answer
    )                                    # end of prompt
    answer = llm(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"].strip()  # generate answer
    return answer, trace                 # return answer and trace

# Cell 8 — Run:




In [ ]:
query = "Compare France vs Germany sales and flag any recalls"  # test question

answer, trace = agentic_rag(query)       # run the agent

print("Agent trace:")                    # print header for trace
for line in trace:                       # loop over trace lines
    print(" ", line)                     # print each trace line

print("\nFinal answer:")                 # print header for answer
print(answer)                            # print the final answer